**Navigation** : [carnet source Python](Chaines-03-Mesure-Lavage-POVM-Python.ipynb) | [Index de la série](README.md)

# Chaînes — Carnet 03 (C#) : La mesure : le lavage à un acte et la POVM

Jumeau C# de [Chaînes-03 — la mesure : le lavage à un acte et la POVM (Python)](Chaines-03-Mesure-Lavage-POVM-Python.ipynb) : même fil pédagogique, mêmes entrées, même arithmétique exacte — rejouée ici en `BigInteger` et en rationnel exact (BCL seule), la trine en entiers d'Eisenstein $\mathbb{Z}[\zeta]$.

Troisième carnet de la série [Chaînes](README.md) — distillation du chapitre 42 du compendium *Chain Theory* (Wissner-Gross, 2026, preprint non relu) : « Measurement: the wash at one act and the POVM ». Après la règle de Born, l'unitarité et la relation d'incertitude (Chaînes-02, chapitres 40-41), le fil passe de **la relation entre deux lectures** à **l'acte d'une lecture** : la mesure, lue ici comme *le lavage à un acte* (*the wash at one act*).

Le théorème 42.1 énonce la lecture collective (*collective readout*) de la toile jointe de plusieurs corps — l'objet que la littérature classe sous la **mesure à opérateurs positifs** (POVM). Comme tous les théorèmes physiques du compendium, il est **conditionnel** : `Standing: a conditional theorem`, adossé à l'axiome et aux résultats des chapitres 1, 18, 28 et 40, chacun à son propre statut.

Le patron du carnet est celui de la série : **Mesuré** (ce que le code rejoue lui-même), **Cité** (toute la chaîne dérivative de la source), **Ouvert** (ce que la source ne prouve pas, listé ligne à ligne — ici la relation de complétude des dix-sept effets, qui **part**).

In [1]:
// Arithmetique exacte : Rationnel (Q en BigInteger) et entiers d'Eisenstein Z[zeta] pour la trine.
// BCL seule : System.Numerics et System.Globalization -- aucun paquet externe, aucun flottant.
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;

// Rationnel exact reduit : numerateur et denominateur en BigInteger, denominateur strictement positif.
sealed class Rationnel
{
    public BigInteger Numerateur { get; }
    public BigInteger Denominateur { get; }

    public Rationnel(BigInteger numerateur, BigInteger denominateur)
    {
        if (denominateur == BigInteger.Zero)
            throw new ArgumentException("Denominateur nul");
        if (denominateur < BigInteger.Zero)
        {
            numerateur = -numerateur;
            denominateur = -denominateur;
        }
        BigInteger pgcd = BigInteger.GreatestCommonDivisor(BigInteger.Abs(numerateur), denominateur);
        Numerateur = numerateur / pgcd;
        Denominateur = denominateur / pgcd;
    }

    public static implicit operator Rationnel(int n) => new Rationnel(n, BigInteger.One);
    public static Rationnel Entier(BigInteger n) => new Rationnel(n, BigInteger.One);

    public static Rationnel operator +(Rationnel x, Rationnel y) =>
        new Rationnel(x.Numerateur * y.Denominateur + y.Numerateur * x.Denominateur,
                      x.Denominateur * y.Denominateur);
    public static Rationnel operator -(Rationnel x, Rationnel y) =>
        new Rationnel(x.Numerateur * y.Denominateur - y.Numerateur * x.Denominateur,
                      x.Denominateur * y.Denominateur);
    public static Rationnel operator -(Rationnel x) => new Rationnel(-x.Numerateur, x.Denominateur);
    public static Rationnel operator *(Rationnel x, Rationnel y) =>
        new Rationnel(x.Numerateur * y.Numerateur, x.Denominateur * y.Denominateur);
    public static Rationnel operator /(Rationnel x, Rationnel y) =>
        new Rationnel(x.Numerateur * y.Denominateur, x.Denominateur * y.Numerateur);

    public static bool operator ==(Rationnel x, Rationnel y) =>
        x.Numerateur == y.Numerateur && x.Denominateur == y.Denominateur;
    public static bool operator !=(Rationnel x, Rationnel y) => !(x == y);

    public override bool Equals(object obj) => obj is Rationnel r && this == r;
    public override int GetHashCode() => Numerateur.GetHashCode() ^ (Denominateur.GetHashCode() * 31);
    public override string ToString() =>
        Denominateur == BigInteger.One ? Numerateur.ToString() : $"{Numerateur}/{Denominateur}";

    public bool EstNul => Numerateur == BigInteger.Zero;
}

// Entier d'Eisenstein a + b*zeta avec zeta = exp(i 2 pi / 3) : zeta^2 = -1 - zeta, zeta^3 = 1.
// Conjugue : zeta_bar = zeta^2. Norme N(a + b zeta) = a^2 - a b + b^2 (entier >= 0, nul ssi nul).
readonly struct Eis
{
    public BigInteger A { get; }
    public BigInteger B { get; }
    public Eis(BigInteger a, BigInteger b) { A = a; B = b; }
    public Eis(int a, int b) { A = a; B = b; }
    public static Eis Zero => new Eis(0, 0);
    public static Eis Un => new Eis(1, 0);
    public static Eis operator +(Eis x, Eis y) => new Eis(x.A + y.A, x.B + y.B);
    public static Eis operator -(Eis x, Eis y) => new Eis(x.A - y.A, x.B - y.B);
    public static Eis operator -(Eis x) => new Eis(-x.A, -x.B);
    // (a + b z)(c + d z) = (ac - bd) + (ad + bc - bd) z   [z^2 = -1 - z]
    public static Eis operator *(Eis x, Eis y) =>
        new Eis(x.A * y.A - x.B * y.B, x.A * y.B + x.B * y.A - x.B * y.B);
    public Eis Conjugue() => new Eis(A - B, -B);
    public BigInteger Norme() => A * A - A * B + B * B;
    public bool EstNul => A.IsZero && B.IsZero;
}

// Renforcement du port : toute conformite est une assertion EXECUTABLE qui leve si la valeur differe.
// La source Python imprime des booleens calcules ; le jumeau C# ne s'arrete pas a l'impression.
static class Exige
{
    public static void Que(bool condition, string label)
    {
        Console.WriteLine($"exige {label} : {(condition ? "CONFORME" : "ECART")}");
        if (!condition)
            throw new InvalidOperationException($"Ecart : {label}");
    }
}

Console.WriteLine($".NET {Environment.Version} -- arithmetique exacte : Rationnel (Q), Z[zeta] pour la trine");

.NET 10.0.15 -- arithmetique exacte : Rationnel (Q), Z[zeta] pour la trine


## Le corps de la lecture collective

La plus petite toile jointe de deux corps du chapitre 42 est **deux copies du corps à deux
docks** (le corps de la lecture collective), appariées à un dock. Ce corps compte **neuf
sites** — trois qutrits $A$, $B$, $C$, un site $H$ de dimension 4, trois qubits $a_1$, $a_2$,
$a_3$ et deux qubits de dock $w_1$, $w_2$ — et **dix liens** notés par leurs deux sites
d'extrémité. La **mesure d'un lien** est le produit des dimensions de ses deux extrémités :
9, 12, 8, 4 et 6 aux cinq types $(3,3)$, $(3,4)$, $(2,4)$, $(2,2)$, $(2,3)$.

La source invite elle-même le lecteur à rejouer l'arithmétique : *« A reader reproduces the
four counts with a few lines of arithmetic from the link list above »*. C'est la partie
**Mesuré** la plus directe du chapitre : construire la liste des liens depuis l'énoncé, en
vérifier les cinq mesures, puis retrouver les premières puissances (éq. 59) et les quatre
comptes de paires (éq. 60) — chaque nombre un entier exact.


In [2]:
// Corps a deux docks (ch. 42, p. 118) : neuf sites, dix liens, cinq types.
var dims = new Dictionary<string, BigInteger> {
    ["A"] = 3, ["B"] = 3, ["C"] = 3, ["H"] = 4,
    ["a1"] = 2, ["a2"] = 2, ["a3"] = 2, ["w1"] = 2, ["w2"] = 2
};
var liens = new List<(string, string)> {
    ("A", "B"), ("B", "C"), ("A", "C"), ("H", "A"),
    ("H", "a1"), ("H", "a2"), ("H", "a3"), ("a1", "a2"),
    ("B", "w1"), ("C", "w2")
};
var mesures = new Dictionary<(string, string), BigInteger>();
foreach (var l in liens)
    mesures[l] = dims[l.Item1] * dims[l.Item2];

// Les cinq types de lien et leur mesure (produit des dimensions des extremites).
var types = new Dictionary<(BigInteger, BigInteger), List<(string, string)>>();
foreach (var kv in mesures)
{
    var d1 = dims[kv.Key.Item1];
    var d2 = dims[kv.Key.Item2];
    var cle = d1 >= d2 ? (d1, d2) : (d2, d1);
    if (!types.TryGetValue(cle, out var membres))
    {
        membres = new List<(string, string)>();
        types[cle] = membres;
    }
    membres.Add(kv.Key);
}
foreach (var cle in types.Keys.OrderByDescending(k => k.Item1).ThenByDescending(k => k.Item2))
    Console.WriteLine($"type ({cle.Item1},{cle.Item2}) : mesure {cle.Item1 * cle.Item2} -> [{string.Join(", ", types[cle].Select(t => t.Item1 + t.Item2).OrderBy(s => s))}]");
Console.WriteLine($"types distincts : {types.Count}");
Exige.Que(liens.Count == 10, "dix liens");
Exige.Que(types.Count == 5, "cinq types de lien distincts");
foreach (var cle in types)
{
    var premier = types[cle.Key].First();
    Exige.Que(mesures[premier] == cle.Key.Item1 * cle.Key.Item2,
              $"mesure du type ({cle.Key.Item1},{cle.Key.Item2}) = produit des dimensions");
}

type (4,3) : mesure 12 -> [HA]


type (4,2) : mesure 8 -> [Ha1, Ha2, Ha3]


type (3,3) : mesure 9 -> [AB, AC, BC]


type (3,2) : mesure 6 -> [Bw1, Cw2]


type (2,2) : mesure 4 -> [a1a2]


types distincts : 5


... (3 outputs left out)

exige mesure du type (4,3) = produit des dimensions : CONFORME


exige mesure du type (4,2) = produit des dimensions : CONFORME


exige mesure du type (2,2) = produit des dimensions : CONFORME


exige mesure du type (3,2) = produit des dimensions : CONFORME


In [3]:
// Eq. 59 : premieres puissances des trois blocs (habillage 9 de la toile jointe a deux copies).
// Liens non frappes d'une copie : les dix liens sauf le lien de dock non apparie Cw2.
BigInteger habillage = 9;
var frappe = ("C", "w2");          // les liens de dock non apparies des deux copies sont frappes
BigInteger sommeNonFrappees = mesures.Where(kv => kv.Key != frappe)
                                     .Aggregate(BigInteger.Zero, (acc, kv) => acc + kv.Value);
BigInteger S1copie = habillage * sommeNonFrappees;
BigInteger S1appariement = habillage * 4;    // lien d'appariement w1w1 de type (2,2), mesure 4
BigInteger S1 = 2 * S1copie + S1appariement;

// Eq. 60 : les quatre comptes de paires de la lecture collective, lecture frappee.
BigInteger own = 2 * S1copie * S1copie;
BigInteger cross = -2 * S1copie * S1copie;
BigInteger crossJoin = 2 * 2 * S1appariement * S1copie;
BigInteger joinJoin = S1appariement * S1appariement;
BigInteger sommeNonSignee = own - cross + crossJoin + joinJoin;

Console.WriteLine($"S1(copie) = {S1copie} (attendu 657)");
Console.WriteLine($"S1(appariement) = {S1appariement} (attendu 36)");
Console.WriteLine($"S1 = {S1} (attendu 1350)");
Console.WriteLine($"own = {own} | cross = {cross} | cross-join = {crossJoin} | join-join = {joinJoin}");
Console.WriteLine($"somme non signee = {sommeNonSignee} = 1350^2 = {BigInteger.Pow(1350, 2)} -> {sommeNonSignee == BigInteger.Pow(1350, 2)}");

// Conformite aux valeurs de la source (eq. 59 et eq. 60) : huit assertions EXECUTABLES derivees.
Exige.Que(S1copie == 657, "S1(copie) = 657");
Exige.Que(S1appariement == 36, "S1(appariement) = 36");
Exige.Que(S1 == 1350, "S1 = 1350");
Exige.Que(own == 863298, "own = 863298");
Exige.Que(cross == -863298, "cross = -863298");
Exige.Que(crossJoin == 94608, "cross-join = 94608");
Exige.Que(joinJoin == 1296, "join-join = 1296");
Exige.Que(sommeNonSignee == BigInteger.Pow(1350, 2), "somme non signee = 1350^2 = 1822500");

S1(copie) = 657 (attendu 657)


S1(appariement) = 36 (attendu 36)


S1 = 1350 (attendu 1350)


own = 863298 | cross = -863298 | cross-join = 94608 | join-join = 1296


somme non signee = 1822500 = 1350^2 = 1822500 -> True


exige S1(copie) = 657 : CONFORME


... (3 outputs left out)

exige cross = -863298 : CONFORME


exige cross-join = 94608 : CONFORME


exige join-join = 1296 : CONFORME


exige somme non signee = 1350^2 = 1822500 : CONFORME


### Lecture du résultat

La liste des liens est celle de l'énoncé, mot pour mot : dix liens, cinq types, et la mesure de
chaque type est bien le produit des dimensions de ses extrémités — $(3,3)\to 9$, $(3,4)\to 12$,
$(2,4)\to 8$, $(2,2)\to 4$, $(2,3)\to 6$. Les premières puissances (éq. 59) ne sont pas seulement
affirmées : elles sont confrontées aux valeurs de la source par la machine, et les **huit assertions
exécutables passent** — elles lèvent si une valeur diffère — $S_1(\text{copie}) = 657$ (habillage 9
fois les 73 d'un lien frappé en moins), $S_1(\text{appariement}) = 36$, $S_1 = 1350$. Les quatre
comptes de paires (éq. 60, figure 17) sont confrontés de même à leurs valeurs source : 863298,
−863298, 94608 et 1296 — les **termes croisés portent le signe moins**. La somme non signée
$1822500 = 1350^2$ se referme au carré de la première puissance, comme l'énonce la source.
C'est l'arithmétique exacte que le chapitre 42 invitait le lecteur à reproduire : ici chaque
conformité est **Mesurée** par le code en entiers exacts, assertion exécutable à l'appui.

## L'espace des enregistrements de la paire (éq. 58)

La lecture collective vit sur l'**espace des enregistrements de la paire** — éq. 58, partie
exacte du chapitre. D'après le type de paire à deux qubits lu à la polarité du slot et au
spectre des enregistrements, la source énonce une **garnison de cinquante-quatre** : six
enregistrements fixes par échange, vingt-quatre orbites, et quatre secteurs tenant **24
enregistrements impairs, 18 pairs mixtes, 12 pairs identiques et 3 de clôture** — cinquante-sept
enregistrements en tout, la face de Born **96**, le tout seize et le coupé neuf, avec trente et un
termes à un échange et trois à deux échanges. La somme à deux échanges est un élément exact de
l'anneau :

$$G_2(2,2) = 5216 + 5216\sqrt{2}, \qquad N = 24+18+12+3 = 57, \qquad Z = 96.$$

La garnison elle-même et la valeur $G_2$ sont **Citées** — leur construction remonte aux
chapitres 18 et 28, hors du périmètre de ce carnet. Ce qui se vérifie ici est l'arithmétique
annoncée : les quatre secteurs somment bien à $N = 57$.


In [4]:
// Eq. 58 : les quatre secteurs de l'espace des enregistrements (valeurs CITEES, addition reverifiee).
var secteurs = new Dictionary<string, BigInteger> {
    ["impairs"] = 24, ["pairs mixtes"] = 18, ["pairs identiques"] = 12, ["cloture"] = 3
};
BigInteger N = secteurs.Values.Aggregate(BigInteger.Zero, (a, v) => a + v);
BigInteger Z = 96;                                    // face de Born, citee
// G_2(2,2) = 5216 + 5216 sqrt(2) : CITE. Represente exactement comme couple de Z[sqrt(2)] --
// la racine n'est jamais substituee par un flottant : la forme exacte seule est imprimee.
var G2 = (r: (BigInteger)5216, s: (BigInteger)5216);
foreach (var kv in secteurs)
    Console.WriteLine($"secteur {kv.Key} : {kv.Value}");
Console.WriteLine($"N = {N} (cite : 57) -> {N == 57}");
Console.WriteLine($"Z = {Z} (face de Born, citee)");
Console.WriteLine($"G_2(2,2) = {G2.r} + {G2.s}*sqrt(2) (cite, forme exacte de Z[sqrt(2)])");
Exige.Que(N == 57, "les quatre secteurs somment a N = 57");

secteur impairs : 24


secteur pairs mixtes : 18


secteur pairs identiques : 12


secteur cloture : 3


N = 57 (cite : 57) -> True


Z = 96 (face de Born, citee)


G_2(2,2) = 5216 + 5216*sqrt(2) (cite, forme exacte de Z[sqrt(2)])


exige les quatre secteurs somment a N = 57 : CONFORME


### Lecture du résultat

L'addition se referme : $24 + 18 + 12 + 3 = 57$ enregistrements, comme l'énonce l'éq. 58. Le
reste — la garnison de cinquante-quatre, les six fixes par échange, les vingt-quatre orbites,
la valeur exacte $G_2(2,2) = 5216 + 5216\sqrt 2$, la face de Born 96 — est cité tel quel : sa
construction remonte aux chapitres 18 et 28, et ce carnet ne la rejoue pas. Ce que le code
prouve ici est uniquement que les quatre secteurs annoncés somment au $N$ annoncé — une
vérification d'arithmétique, pas une dérivation.


## Le swap d'intrication : distiller au qudit partagé

Le mécanisme que la source met au centre du théorème 42.1 est la **toile debout** : *« one
stabilizer state with two generators per link »* — et sa distillation : *« distilling at every
shared qudit is the entanglement swap, leaving the far ends a pair on their common support at
every outcome [...] with a Weyl letter as the byproduct »*. La correspondance avec la
littérature est nommée : l'échange d'intrication projette deux systèmes sur une paire
maximalement intriquée et laisse leurs partenaires intriqués à un opérateur de sous-produit
près [22].

C'est rejouable ici, à l'extrémité qubit, en quelques lignes : deux paires de Bell $AB$ et $CD$
dans $\lvert\omega\rangle = (\lvert 00\rangle + \lvert 11\rangle)/\sqrt 2$, une lecture du couple
central $BC$ dans la **base de Bell** — quatre issues, une par lettre de Weyl $W \in \{I, X, Z,
XZ\}$, chacune $\lvert\!\mathrm{bell}\rangle = (I \otimes W)\lvert\omega\rangle$ (les lettres de
Chaînes-02) — et la question : dans quel état tombe le couple éloigné $AD$ ?


In [5]:
// Swap d'intrication a l'extremite qubit : deux paires de Bell, lecture de BC en base de Bell.
// Representation exacte : le facteur 1/sqrt(2) de |omega> est elude par invariance d'echelle --
// vecteurs 16-dim et matrices 16x16 en BigInteger, normalises par les normes entieres :
//   p reel = <phi|phi> / <psi0|psi0>   avec psi0 = |~omega> (x) |~omega>, <psi0|psi0> = 4.
public static class SwapOutils
{
public static BigInteger[][] MatIdentite(int n)
{
    var M = new BigInteger[n][];
    for (int i = 0; i < n; i++) { M[i] = new BigInteger[n]; M[i][i] = 1; }
    return M;
}
public static BigInteger[][] MatKron(BigInteger[][] A, BigInteger[][] B)
{
    int n = A.Length * B.Length;
    var M = new BigInteger[n][];
    for (int i = 0; i < n; i++) M[i] = new BigInteger[n];
    for (int a = 0; a < A.Length; a++)
        for (int b = 0; b < B.Length; b++)
            for (int c = 0; c < A.Length; c++)
                for (int d = 0; d < B.Length; d++)
                    M[a * B.Length + b][c * B.Length + d] = A[a][c] * B[b][d];
    return M;
}
public static BigInteger[] VecKron(BigInteger[] u, BigInteger[] v)
{
    var w = new BigInteger[u.Length * v.Length];
    for (int i = 0; i < u.Length; i++)
        for (int j = 0; j < v.Length; j++)
            w[i * v.Length + j] = u[i] * v[j];
    return w;
}
public static BigInteger[] Appliquer(BigInteger[][] M, BigInteger[] v)
{
    var w = new BigInteger[M.Length];
    for (int i = 0; i < M.Length; i++)
        for (int j = 0; j < v.Length; j++)
            w[i] += M[i][j] * v[j];
    return w;
}
public static BigInteger Dot(BigInteger[] u, BigInteger[] v)
{
    BigInteger s = 0;
    for (int i = 0; i < u.Length; i++) s += u[i] * v[i];
    return s;
}
public static BigInteger[][] Exterieur(BigInteger[] v)
{
    var M = new BigInteger[v.Length][];
    for (int i = 0; i < v.Length; i++)
    {
        M[i] = new BigInteger[v.Length];
        for (int j = 0; j < v.Length; j++) M[i][j] = v[i] * v[j];
    }
    return M;
}
public static BigInteger[][] MulMat(BigInteger[][] A, BigInteger[][] B)
{
    var M = new BigInteger[A.Length][];
    for (int i = 0; i < A.Length; i++)
    {
        M[i] = new BigInteger[B[0].Length];
        for (int j = 0; j < B[0].Length; j++)
            for (int k = 0; k < A.Length; k++)
                M[i][j] += A[i][k] * B[k][j];
    }
    return M;
}
public static BigInteger[][] SommeMat(BigInteger[][] A, BigInteger[][] B)
{
    var M = new BigInteger[A.Length][];
    for (int i = 0; i < A.Length; i++)
    {
        M[i] = new BigInteger[A.Length];
        for (int j = 0; j < A.Length; j++) M[i][j] = A[i][j] + B[i][j];
    }
    return M;
}
public static BigInteger TrMat(BigInteger[][] A)
{
    BigInteger s = 0;
    for (int i = 0; i < A.Length; i++) s += A[i][i];
    return s;
}
public static bool EgalMat(BigInteger[][] A, BigInteger[][] B)
{
    for (int i = 0; i < A.Length; i++)
        for (int j = 0; j < A.Length; j++)
            if (A[i][j] != B[i][j]) return false;
    return true;
}
}

var omegaT = new BigInteger[] { 1, 0, 0, 1 };              // |~omega> = |00> + |11> (non normalise)
var I2 = SwapOutils.MatIdentite(2);
var X = new BigInteger[][] { new BigInteger[] { 0, 1 }, new BigInteger[] { 1, 0 } };
var Zm = new BigInteger[][] { new BigInteger[] { 1, 0 }, new BigInteger[] { 0, -1 } };
var lettres = new Dictionary<string, BigInteger[][]> {
    ["I"] = I2, ["X"] = X, ["Z"] = Zm, ["XZ"] = SwapOutils.MulMat(X, Zm)
};

// Quatre issues de la lecture, une par lettre de Weyl : bell_W = (I (x) W)|~omega>.
var bellT = new Dictionary<string, BigInteger[]>();
foreach (var kv in lettres)
    bellT[kv.Key] = SwapOutils.Appliquer(SwapOutils.MatKron(I2, kv.Value), omegaT);

// Base orthonormee a l'echelle 2 : la somme des projecteurs non normalises vaut exactement 2*I_4.
var sommeProj = new BigInteger[4][] { new BigInteger[4], new BigInteger[4], new BigInteger[4], new BigInteger[4] };
foreach (var v in bellT.Values) sommeProj = SwapOutils.SommeMat(sommeProj, SwapOutils.Exterieur(v));
var deuxIdentite = SwapOutils.MatIdentite(4);
for (int i = 0; i < 4; i++) deuxIdentite[i][i] = 2;
Console.WriteLine($"les 4 etats de Bell forment une base orthonormee : {SwapOutils.EgalMat(sommeProj, deuxIdentite)}");
Exige.Que(SwapOutils.EgalMat(sommeProj, deuxIdentite), "somme des projecteurs de Bell = 2*I_4 (base orthonormee)");

// Etat de depart : |~omega>_AB (x) |~omega>_CD sur ABCD ; lecture projective de BC.
var psi0T = SwapOutils.VecKron(omegaT, omegaT);
Console.WriteLine("\nlettre W | proba | purete | trace partielle = I/2 | fidelite avec |bell_W>");
foreach (var kv in lettres)
{
    var v = bellT[kv.Key];
    var P_BC = SwapOutils.MatKron(SwapOutils.MatKron(I2, SwapOutils.Exterieur(v)), I2);      // projecteur sur BC seul, etendu
    var phi = SwapOutils.Appliquer(P_BC, psi0T);                        // etat (non normalise) de ABCD
    BigInteger p = SwapOutils.Dot(phi, phi);                            // <psi0|psi0> = 4 : proba = p/4
    var rhoChap = SwapOutils.Exterieur(phi);

    // Trace partielle sur B et C : apparier (B,B') et (C,C') par lettres partagees b et c.
    var MChap = new BigInteger[4][] { new BigInteger[4], new BigInteger[4], new BigInteger[4], new BigInteger[4] };
    for (int a = 0; a < 2; a++)
        for (int d = 0; d < 2; d++)
            for (int e = 0; e < 2; e++)
                for (int h = 0; h < 2; h++)
                {
                    BigInteger s = 0;
                    for (int b = 0; b < 2; b++)
                        for (int c = 0; c < 2; c++)
                            s += rhoChap[8 * a + 4 * b + 2 * c + d][8 * e + 4 * b + 2 * c + h];
                    MChap[2 * a + d][2 * e + h] = s;
                }

    // Purete Tr(rho_AD^2) = Tr(MChap^2) / p^2 : rationnel exact.
    var purete = Rationnel.Entier(SwapOutils.TrMat(SwapOutils.MulMat(MChap, MChap))) / Rationnel.Entier(p * p);

    // Trace partielle sur D : MChap_A[a][e] = somme_d MChap[(a,d),(e,d)], attendue (p/2)*I_2 entrée par entrée.
    var MA = new BigInteger[2][] { new BigInteger[2], new BigInteger[2] };
    for (int a = 0; a < 2; a++)
        for (int e = 0; e < 2; e++)
            MA[a][e] = MChap[2 * a + 0][2 * e + 0] + MChap[2 * a + 1][2 * e + 1];
    bool plat = MA[0][0] == p / 2 && MA[1][1] == p / 2 && MA[0][1] == 0 && MA[1][0] == 0;

    // Fidelite <v|rho_AD|v> = <~v|MChap|~v> / (2 p) : rationnel exact.
    var fid = Rationnel.Entier(SwapOutils.Dot(v, SwapOutils.Appliquer(MChap, v))) / (Rationnel.Entier(2) * Rationnel.Entier(p));

    var proba = new Rationnel(p, 4);
    Console.WriteLine($"    {kv.Key,2}   | {proba}  |  {purete}  |         {plat}          | {fid}");
    Exige.Que(p == 4, $"proba de l'issue {kv.Key} : echelle entiere 4 (= 1/4 en proba)");
    Exige.Que(purete == 1, $"purete {kv.Key} : Tr(rho_AD^2) = 1 exactement");
    Exige.Que(plat, $"trace partielle {kv.Key} : rho_A = I/2 entrée par entrée (MChap_A = (p/2) I_2)");
    Exige.Que(fid == 1, $"fidelite {kv.Key} avec |bell_{kv.Key}> : exactement 1");
}

les 4 etats de Bell forment une base orthonormee : True


exige somme des projecteurs de Bell = 2*I_4 (base orthonormee) : CONFORME



lettre W | proba | purete | trace partielle = I/2 | fidelite avec |bell_W>


     I   | 1  |  1  |         True          | 1


exige proba de l'issue I : echelle entiere 4 (= 1/4 en proba) : CONFORME


exige purete I : Tr(rho_AD^2) = 1 exactement : CONFORME


... (13 outputs left out)

exige proba de l'issue XZ : echelle entiere 4 (= 1/4 en proba) : CONFORME


exige purete XZ : Tr(rho_AD^2) = 1 exactement : CONFORME


exige trace partielle XZ : rho_A = I/2 entrée par entrée (MChap_A = (p/2) I_2) : CONFORME


exige fidelite XZ avec |bell_XZ> : exactement 1 : CONFORME


### Lecture du résultat

À chacune des quatre issues de la lecture du couple central — une issue par lettre de Weyl — le
couple éloigné $AD$ tombe **exactement** dans l'état de Bell qui porte la même lettre : pureté
$\mathrm{Tr}(\rho^2) = 1$ **en rationnel exact**, trace partielle $I/2$ vérifiée entrée par entrée
sur l'opérateur non normalisé, et fidélité **exactement $1$** avec $\lvert\mathrm{bell}_W\rangle$.
Chaque issue a la probabilité $1/4$ (exacte, à l'échelle entière), et le sous-produit de la
distillation est la lettre $W$ elle-même. C'est le geste que le compendium décrit à chaque qudit
partagé (*« with a Weyl letter as the byproduct »*) — rejoué ici à l'extrémité qubit de la
construction. Les deux paires de départ ne se touchaient pas : l'intrication entre $A$ et $D$ est
**créée par l'acte de lecture seul** — c'est le sens de *the wash at one act*.

## La POVM côté littérature — le référentiel du gloss

Le gloss du chapitre 42 s'appuie sur la définition de la littérature : une **mesure généralisée**
est un ensemble d'opérateurs positifs $E_i$ sommant à l'identité, avec probabilités d'issue
$\mathrm{Tr}(\rho E_i)$ — et toute mesure généralisée se relève, par le théorème de dilatation,
en une mesure projective sur un espace plus grand [74]. C'est le référentiel **exact** contre
lequel la source fait partir sa propre construction : la question n'est pas de savoir si des
opérateurs positifs existent, mais si leur somme **est** l'identité du registre.

Pour voir ce que « complète » veut dire, rien ne vaut une POVM qui l'est : la **trine** — trois
effets $E_k = \tfrac{2}{3}\lvert\psi_k\rangle\langle\psi_k\rvert$ aux kets séparés de 120° sur
l'équateur du Bloch. La somme des effets est l'identité **par construction**, et les probabilités
$\mathrm{Tr}(\rho E_k)$ somment à 1 pour **tout** état — pur ou mixte — ce qui est exactement le
contenu de la relation de complétude.


In [6]:
// La trine : trois effets positifs sur un qubit, complete par construction.
// Ket |psi_k> = (|0> + zeta^k |1>)/sqrt(2), zeta = exp(i 2 pi / 3) : represente NON NORMALISE en
// Z[zeta] (norme entiere N = 2), le facteur 1/sqrt(2) etant elude par invariance d'echelle.
// Element de Q(zeta) : A + B*zeta avec A, B rationnels exacts.
readonly struct EisQ
{
    public Rationnel A { get; }
    public Rationnel B { get; }
    public EisQ(Rationnel a, Rationnel b) { A = a; B = b; }
    public EisQ(Rationnel a) { A = a; B = 0; }
    public static EisQ operator +(EisQ x, EisQ y) => new EisQ(x.A + y.A, x.B + y.B);
    public static EisQ operator -(EisQ x, EisQ y) => new EisQ(x.A - y.A, x.B - y.B);
    public static EisQ operator *(EisQ x, EisQ y) =>
        new EisQ(x.A * y.A - x.B * y.B, x.A * y.B + x.B * y.A - x.B * y.B);
    public static bool operator ==(EisQ x, EisQ y) => x.A == y.A && x.B == y.B;
    public static bool operator !=(EisQ x, EisQ y) => !(x == y);
    public static EisQ operator *(Rationnel x, EisQ y) => new EisQ(x * y.A, x * y.B);
    public override bool Equals(object obj) => obj is EisQ q && this == q;
    public override int GetHashCode() => A.GetHashCode() ^ (B.GetHashCode() * 31);
    public EisQ Conjugue() => new EisQ(A - B, -B);
}
static EisQ VersQ(Eis z) => new EisQ(Rationnel.Entier(z.A), Rationnel.Entier(z.B));

static class TrineOutils
{
    public static Eis ZetaPuissance(int k)
    {
        k = ((k % 3) + 3) % 3;
        return k == 0 ? Eis.Un : (k == 1 ? new Eis(0, 1) : new Eis(-1, -1));
    }
}

var ketT = Enumerable.Range(0, 3).Select(k => new[] { Eis.Un, TrineOutils.ZetaPuissance(k) }).ToArray();

// Effets E_k = (2/3)|psi_k><psi_k| : a l'echelle non normalisee, E_k[i][j] = (1/3) * psi~[i] * conj(psi~[j]).
var tiers = new Rationnel(1, 3);
var effets = new EisQ[3][][];
for (int k = 0; k < 3; k++)
{
    effets[k] = new EisQ[2][];
    for (int i = 0; i < 2; i++)
    {
        effets[k][i] = new EisQ[2];
        for (int j = 0; j < 2; j++)
            effets[k][i][j] = tiers * (VersQ(ketT[k][i]) * VersQ(ketT[k][j]).Conjugue());
    }
}

// Completude : la somme des trois effets est EXACTEMENT l'identite I_2 (egalite entree par entree).
var somme = new EisQ[2][] { new EisQ[2], new EisQ[2] };
for (int i = 0; i < 2; i++)
    for (int j = 0; j < 2; j++)
        somme[i][j] = effets[0][i][j] + effets[1][i][j] + effets[2][i][j];
Console.WriteLine("somme des effets == I_2 :", somme[0][0] == new EisQ(1) && somme[1][1] == new EisQ(1)
                                            && somme[0][1].A.EstNul && somme[0][1].B.EstNul
                                            && somme[1][0].A.EstNul && somme[1][0].B.EstNul);
Exige.Que(somme[0][0] == new EisQ(1) && somme[1][1] == new EisQ(1)
          && somme[0][1].A.EstNul && somme[0][1].B.EstNul
          && somme[1][0].A.EstNul && somme[1][0].B.EstNul,
          "somme des effets = I_2 exactement (completude par construction)");

// Hermitien et positif : hermiticite par egalite exacte E[i][j] = conj(E[j][i]) ; positivite par
// CERTIFICAT de rang 1 (remplace eigvalsh flottant) : det(E_k) = 0 et trace(E_k) = 2/3 donnent
// le spectre exact {0, 2/3}, positif -- un operateur 2x2 hermitien de determinant nul et de trace
// strictement positive a son unique valeur propre non nulle egale a la trace.
for (int k = 0; k < 3; k++)
{
    bool herm = effets[k][0][1] == effets[k][1][0].Conjugue();
    var det = effets[k][0][0] * effets[k][1][1] - effets[k][0][1] * effets[k][1][0];
    var trace = effets[k][0][0] + effets[k][1][1];
    var deuxTiers = new EisQ(new Rationnel(2, 3));
    Console.WriteLine($"effet {k} : hermitien {herm}, det = {det.A} + ({det.B})z, trace = {trace.A}");
    Exige.Que(herm, $"effet {k} hermitien (egalite exacte)");
    Exige.Que(det.A.EstNul && det.B.EstNul, $"det(E_{k}) = 0 (rang 1)");
    Exige.Que(trace == deuxTiers, $"trace(E_{k}) = 2/3 -> spectre exact {{0, 2/3}}, positif");
}

// Probabilites d'issue Tr(rho E_k) sur trois etats distincts : pur |0>, pur |+>, mixte maximal.
// Pur reel phi~ : p_k = (2/3) * N(<psi~_k|phi~>) / (N(psi~_k) * <phi~|phi~>) -- entiers exacts.
var etats = new (string nom, BigInteger[] phi, bool mixte)[] {
    ("|0> (pur)", new BigInteger[] { 1, 0 }, false),
    ("|+> (pur)", new BigInteger[] { 1, 1 }, false),
    ("I/2 (mixte maximal)", null, true)
};
foreach (var (nom, phi, mixte) in etats)
{
    var probas = new Rationnel[3];
    for (int k = 0; k < 3; k++)
    {
        if (mixte)
        {
            // Tr((I/2) E_k) = trace(E_k)/2 = 1/3 : exact.
            probas[k] = new Rationnel(2, 3) / 2;
        }
        else
        {
            // <psi~_k|phi~> = phi[0] + conj(zeta^k) * phi[1] (phi reel) : element de Z[zeta].
            var inner = new Eis(phi[0], 0) + TrineOutils.ZetaPuissance(k).Conjugue() * new Eis(phi[1], 0);
            BigInteger normePhi = phi[0] * phi[0] + phi[1] * phi[1];
            probas[k] = new Rationnel(2 * inner.Norme(), 3 * 2 * normePhi);
        }
    }
    var sommeProbas = probas[0] + probas[1] + probas[2];
    Console.WriteLine($"{nom,20} : probas [{string.Join(", ", probas.Select(p => p.ToString()))}] somme {sommeProbas}");
    Exige.Que(sommeProbas == 1, $"somme des probas = 1 exactement pour {nom}");
}

somme des effets == I_2 :


exige somme des effets = I_2 exactement (completude par construction) : CONFORME


effet 0 : hermitien True, det = 0 + (0)z, trace = 2/3


exige effet 0 hermitien (egalite exacte) : CONFORME


exige det(E_0) = 0 (rang 1) : CONFORME


exige trace(E_0) = 2/3 -> spectre exact {0, 2/3}, positif : CONFORME


... (10 outputs left out)

           |+> (pur) : probas [2/3, 1/6, 1/6] somme 1


exige somme des probas = 1 exactement pour |+> (pur) : CONFORME


 I/2 (mixte maximal) : probas [1/3, 1/3, 1/3] somme 1


exige somme des probas = 1 exactement pour I/2 (mixte maximal) : CONFORME


### Lecture du résultat

La trine est bien une POVM **complète** : $\sum_k E_k = I_2$ par **égalité exacte** entrée par
entrée, chaque effet est hermitien et positif — le certificat de rang 1 (déterminant nul, trace
$2/3$) donne le spectre exact $\{0, 2/3\}$ — et les probabilités d'issue somment à $1$
**exactement, en rationnel** pour les trois états essayés, pur ou mixte. Aucun état ne peut
violer cette normalisation : c'est une identité d'opérateurs, pas une propriété d'état. Ce sera la
ligne de contraste du **Ouvert** ci-dessous : dans la construction du compendium, la somme des
effets **n'est pas** l'identité du registre — et c'est précisément pourquoi le théorème 42.1 reste
conditionnel et le gloss, un gloss.

## Ce que la source ne prouve pas — ligne à ligne

Le chapitre 42 est explicite sur ses propres limites ; le carnet les rapporte telles quelles,
sans les maquiller en résultats établis.

**Le gloss de la mesure généralisée tient en cinq pas, et un d'eux part.** Quatre pas se
résolvent en théorèmes : le lavage à un acte, le placement (la position d'un tenseur dans un
réseau), le taux (*le carré lavé sur le carré de Born*), et le registre de dock comme tout du
support commun. Mais le cinquième — la **complétude** — part : à la toile jointe de l'électron,
les **dix-sept effets**, chacun un rationnel non négatif exact à l'unité de dock, **sument à la
classe croix au dock, tandis que le registre de dock debout est seize**. La relation de
complétude $\sum_i E_i = I$ part à la fois **à la lecture de Born et au compte**. Les effets
sont construits comme opérateurs sur le registre de dock et leur somme est imprimée, mais
**l'identification de cette somme à l'identité du registre part** — contre la trine ci-dessus,
qui la satisfait par construction. C'est la frontière exacte entre le chapitre et la
littérature qu'il invoque [74].

**Les prémices ouvertes, nommées par la source.** Le théorème 42.1 repose sur les résultats
ouverts 18.5, 18.6, 28.3 et 28.4 — chacun à son propre statut, aucun n'est prouvé au sens du
compendium — en plus des théorèmes conditionnels 18.1-18.4, 28.1-28.2, 28.5 et 40.1. La chaîne
de prémices remonte à l'axiome en douze chapitres au plus : **l'axiome lui-même est posé,
jamais dérivé** (Chaînes-01).

**La lecture qui part au census.** À la lecture nommée dans le census, la lecture collective
**part** : la première part est l'étoile à $N = 4$ corps, **refusée à la borne de valence**.
Le census couvre les six toiles jointes que les deux docks admettent à $N = 2, 3, 4$ — la
lecture est **définie** aux six, mais rien ne dit qu'elle survive au-delà de la borne.

**Preprint non relu.** Le compendium est un *working draft* v0.1, non soumis à relecture ;
ses comptes sont certifiés par son propre paquet machine-checké (« 206 objets énumérés »),
pas par la communauté.


## Ce que ce carnet ne prouve pas

Le code ci-dessus **rejoue** des objets finis — une liste de liens, des comptes de paires, un
swap à l'extrémité qubit, une POVM à trois effets — il **ne démontre** ni le théorème 42.1 ni
aucun de ses maillons. La lecture collective aux six toiles jointes (206 liens au total avec le
corps simple), les **3412 swaps à 1100 qudits partagés sur dix triplets distincts**, le compte
de rondes égal à la profondeur aux **64 toiles**, la sortie comme trace normalisée aux **309
lectures**, le second moteur de reproduction, les contretypes et contrôles : tout cela est
**Cité**, vérifié par la source à ses propres moteurs, jamais rejoué ici. Les dix-sept effets
de l'électron ne sont pas non plus rejoués : leurs valeurs ne sont pas imprimées au chapitre,
seul leur statut l'est (chacun un rationnel non négatif exact, la somme partant de seize).
L'arithmétique que le carnet vérifie est celle que la source invite explicitement le lecteur à
reprendre — rien de plus.


## Croisements

- **Chaînes-02** : le théorème 40.1 (le pli plat) est prémice directe du théorème 42.1, et les
  lettres de Weyl y sont construites — le sous-produit du swap d'ici est l'une d'elles.
- **Chaînes-01** : l'axiome d'intrication, posé et jamais dérivé, tête de la chaîne de douze
  chapitres.
- **Jauge-01/02** : les lois de conservation, elles, sont **théorèmes** (non conditionnels) à la
  source — la mesure est conditionnelle, la conservation est dérivée.
- **Chapitre 43** (indiscernabilité et signe spin-statistique par le twirl) et **chapitre 44**
  (l'énergie comme opérateur de première puissance, éq. 64-67) : candidats naturels pour la
  suite de la série.

## Références

- T. Wissner-Gross, *Chain Theory Compendium* (v0.1 working draft, 2026), chapitre 42
  « Measurement: the wash at one act and the POVM », p. 116-119 — preprint non relu, archivé
  au gisement de la bibliographie, lu avant écriture.
- Références nommées par la source : [22] échange d'intrication avec sous-produit ;
  [74] théorème de dilatation (mesure généralisée ↔ projective sur un espace plus grand) ;
  [75] calcul fondé sur la mesure (carnets de stabilisateur, profondeur = rondes).


## Exercices

### Exercice 1 — Symétrie des docks

La source frappe le lien de dock non apparié $Cw_2$ de chaque copie. Les deux docks $w_1$ et
$w_2$ tiennent des liens de même type $(2,3)$ et même mesure. Vérifier que frapper $Bw_1$ et
d'apparier les seconds docks ($w_2w_2$) donne **les mêmes** premières puissances et les mêmes
quatre comptes de paires — la symétrie des deux docks de la lecture collective.


In [7]:
// Exercice 1 : recalculer eq. 59-60 en frappant Bw1 (au lieu de Cw2) et en apparieant w2w2.
// Etape 1 : construire la liste des liens non frappes avec Bw1 frappe.
// Etape 2 : calculer S1(copie), S1(appariement) et S1.
// Etape 3 : deriver les quatre comptes et comparer a own/cross/cross-join/join-join.
using System.Numerics;

BigInteger? S1Bw1 = null;         // TODO etudiant : habillage 9 * somme des mesures hors Bw1
BigInteger?[] comptesBw1 = null;  // TODO etudiant : own, cross, cross-join (appariement w2w2), join-join
Console.WriteLine("Exercice a completer");

Exercice a completer


### Exercice 2 — Le dictionnaire du sous-produit, lettre par lettre

La table du swap montre la fidélité $1$ pour chaque lettre, mais le sous-produit de la
littérature est une **correspondance** : l'issue $W$ lue sur $BC$ produit l'état
$\lvert\mathrm{bell}_{\bar W}\rangle$ sur $AD$ pour une certaine conjugaison $\bar W$. Reconstruire
le dictionnaire complet : pour chaque lettre $W \in \{I, X, Z, XZ\}$, calculer la fidélité de
$\rho_{AD}$ avec les **quatre** états de Bell, et identifier lequel est sélectionné.

*Indice : comparer $\langle\mathrm{bell}_{W'}|\rho_{AD}|\mathrm{bell}_{W'}\rangle$ sur les
quatre valeurs de $W'$ — la table du carnet s'arrête à la diagonale.*


In [8]:
// Exercice 2 : table de fidelite 4x4 -- lignes = issue lue W sur BC, colonnes = etat W' teste sur AD.
// Etape 1 : boucler sur les quatre issues W et recalculer rho_AD comme dans la cellule du swap.
// Etape 2 : pour chaque W, evaluer la fidelite avec les quatre etats bell[W'].
// Etape 3 : imprimer la table et relever la lettre selectionnee sur chaque ligne.
Rationnel[,] tableFidelite = null;  // TODO etudiant : table[W, W'] = <bell_W'|rho_AD|bell_W'> en rationnel exact
Console.WriteLine("Exercice a completer");

Exercice a completer


### Exercice 3 — Une POVM qui n'est pas complète

Construire trois effets à partir des kets de la trine mais avec un poids $\alpha$ au lieu de
$\tfrac23$ : $E_k = \alpha\,\lvert\psi_k\rangle\langle\psi_k\rvert$. Trouver la valeur de
$\alpha$ qui rend la mesure complète, puis montrer que pour tout autre $\alpha$ l'écart
$\sum_k \mathrm{Tr}(\rho E_k) - 1$ est **le même pour tous les états** — une identité
d'opérateurs, pas une propriété d'état. Mesurer cet écart sur $\lvert 0\rangle$ et sur $I/2$
pour $\alpha = \tfrac12$ et $\alpha = \tfrac{1}{3}$ : c'est l'écart de complétude, en petit —
celui que la source constate à l'échelle du registre de dock.


In [9]:
// Exercice 3 : trine a poids alpha -- mesurer l'ecart de completude pour alpha != 2/3.
// Etape 1 : trouver alpha tel que sum(alpha |psi_k><psi_k|) == I_2 (somme des projecteurs d'abord).
// Etape 2 : pour alpha = 1/2 et 1/3, calculer l'ecart sum_k Tr(rho E_k) - 1 sur |0> et I/2.
// Etape 3 : verifier que l'ecart est independant de l'etat (identite d'operateurs) et l'interpreter.
Rationnel alphaComplet = null;  // TODO etudiant : alpha tel que la somme des effets pese I_2 exactement
Rationnel[,] ecarts = null;     // TODO etudiant : ecart de completude sur |0> et I/2 pour alpha = 1/2 et 1/3
Console.WriteLine("Exercice a completer");

Exercice a completer


## À retenir

1. **La mesure est ici le lavage à un acte** : le théorème 42.1 énonce la lecture collective de
   la toile jointe de plusieurs corps, l'objet que la littérature classe sous la POVM — et il
   est **conditionnel**, comme toute la physique du compendium.
2. **L'arithmétique exacte est vérifiable** : corps à neuf sites et dix liens, $S_1 = 1350$, et
   les quatre comptes de paires 863298, $-863298$, 94608, 1296 refermés par
   $1350^2 = 1\,822\,500$ — l'invitation explicite de la source, honorée.
3. **Le mécanisme est le swap** : à chaque qudit partagé, distiller = échanger l'intrication,
   le couple éloigné tombe sur une paire maximalement intriquée **à chaque issue**, avec une
   lettre de Weyl en sous-produit — rejoué ici en fidélité exacte aux quatre issues.
4. **La complétude est la frontière** : la trine satisfait $\sum_i E_i = I$ par construction ;
   les dix-sept effets de la toile de l'électron sument à la classe croix, pas à seize — le
   registre debout. Le gloss de la mesure généralisée tient en cinq pas, et c'est le
   cinquième qui part.
5. **Rien de tout cela n'est établi au sens de la communauté** : prémices ouvertes 18.5, 18.6,
   28.3, 28.4 en amont, étoile refusée à la borne de valence à $N=4$, preprint non relu.


## Vérifiez votre compréhension

1. Quels sont les cinq types de lien du corps à deux docks, et quelle est la mesure de chacun ?
2. Dans l'équation 60, deux des quatre comptes de paires sont négatifs ; pourquoi la somme
   « non signée » se referme-t-elle pourtant sur $1350^2$ ?
3. Après la lecture du couple central $BC$, que valent la trace partielle et la fidélité de
   l'état de $AD$ ? Que disent ces deux nombres sur l'intrication créée par la lecture ?
4. Pourquoi la trine est-elle une POVM complète, alors que la construction du compendium ne
   l'est pas ?

<details>
<summary>Réponses</summary>

1. $(3,3) \to 9$, $(3,4) \to 12$, $(2,4) \to 8$, $(2,2) \to 4$, $(2,3) \to 6$ : la mesure d'un
   lien est le produit des dimensions de ses deux extrémités.
2. Parce que les termes croisés portent le signe moins et que la somme les **retranche** :
   $863298 - (-863298) + 94608 + 1296 = 1\,822\,500 = 1350^2$.
3. Trace partielle $I/2$ et fidélité $1{,}000000000000$ avec $\lvert\mathrm{bell}_W\rangle$ :
   $AD$ est dans une paire maximalement intriquée, alors que les deux paires de départ ne se
   touchaient pas. L'intrication entre $A$ et $D$ est créée par l'acte de lecture seul.
4. La trine somme à l'identité **par construction** ($\sum_k E_k = I_2$), donc ses probabilités
   somment à $1$ pour tout état. Dans la construction de la source, les dix-sept effets de la
   toile de l'électron sument à la classe croix, pas à seize : la somme des effets n'est pas
   l'identité du registre, et c'est pourquoi le théorème 42.1 reste conditionnel.

</details>


## Pour aller plus loin

[Chaînes-04 — Indiscernabilité et signe spin-statistique](Chaines-04-Indiscernabilite-Spin-Statistique-Python.ipynb)
prend la suite du fil : l'indiscernabilité et le signe spin-statistique par le twirl (chapitre 43
du compendium), puis l'énergie comme opérateur de première puissance (chapitre 44). Le geste de
ce carnet y revient : ce que la source dérive et ce qu'elle suppose y sont séparés de la même
manière.

Pour revenir en amont, [Chaînes-02 — la règle de Born, l'unitarité et l'incertitude](Chaines-02-Born-Unitarite-Incertitude-Python.ipynb)
construit les lettres de Weyl qui servent ici de sous-produit au swap, et [Chaînes-01 — l'axiome
d'intrication](Chaines-01-Axiome-Intrication-Constante-Structure-Fine-Python.ipynb) pose l'axiome
dont tout le fil descend.

Hors de la série, [Jauge-01 — le théorème de Noether](../Jauge/Jauge-01-Theoreme-Noether-Python.ipynb) offre le contraste
le plus utile : les lois de conservation y sont des théorèmes, non conditionnels, là où la mesure
est ici conditionnelle à ses prémisses ouvertes. Le chapitre 42 s'appuie enfin sur le théorème de
dilatation de la mesure généralisée [74] et sur les carnets de stabilisateur [75] : deux lectures
qui prolongent la POVM côté littérature.
